# 03A — Optional output review

This read-only notebook prints the small durable outputs from canonicalisation,
EDA and the evaluation harness. It is optional and never changes pipeline state.


In [ ]:
import importlib
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT")
    or os.getenv("ANOMALY_DRIVE_ROOT")
    or ("/content/drive/MyDrive/anomaly_detection" if IN_COLAB
        else Path.home() / "anomaly_detection_data")
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")  # or "petrobras_3w"
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_10_1_run1",
    "petrobras_3w": "petrobras_3w_core_v0_10_1_run2",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json

CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / CANONICAL_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
EVAL_ROOT = RUN_ROOT / "SPEC-EVAL"
SPLIT_ROOT = RUN_ROOT / "SPLITS"

VERSION = "2.0.0"
EDA_ROOT = DATA_ROOT / "outputs" / "eda" / f"v{VERSION}" / SECTOR / f"{SECTOR}_eda_v2_run1"
EVALUATION_ROOT = DATA_ROOT / "outputs" / "evaluation" / f"v{VERSION}" / SECTOR / f"{SECTOR}_evaluation_v2_run1"

display(pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet"))
display(pd.read_csv(EDA_ROOT / "metric_summary.csv"))
display(pd.read_csv(EDA_ROOT / "temporal_summary.csv"))
display(pd.read_parquet(EDA_ROOT / "readiness.parquet").groupby(
    ["channel", "status"]
).size().rename("entity_metric_series").to_frame())
display(pd.Series(read_json(EVALUATION_ROOT / "evaluation_policy.json"), name="value").to_frame())
print("Holdout remains physically separate at:", EVALUATION_ROOT / "holdout_sealed")
